# Midi generation, time signatures

In [ ]:
import pretty_midi
import random
import os

def generate_pro_midi(loop_length_bars=1, tempo=120, numerator=4, denominator=4, filename="temp.mid"):
    pm = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    
    # 1. Set the Time Signature in the MIDI file
    ts = pretty_midi.TimeSignature(numerator, denominator, 0.0)
    pm.time_signature_changes.append(ts)
    
    drums = pretty_midi.Instrument(program=0, is_drum=True)
    
    # We resolve everything to 16th notes
    steps_per_beat = int(16 / denominator) 
    steps_per_bar = numerator * steps_per_beat
    total_steps = steps_per_bar * loop_length_bars
    
    step_time = (60 / tempo) / 4.0
    jitter_amount = 0.05 # ~8ms of human error
    
    # 2. Dynamic Backbeat Logic (Where does the Snare go?)
    if numerator == 3:
        # In 3/4, the backbeat is usually beat 3
        backbeats = [steps_per_beat * 2] 
    elif numerator == 5:
        # In 5/4, let's put backbeats on beats 2 and 4
        backbeats = [steps_per_beat * 1, steps_per_beat * 3]
    else: 
        # Default 4/4 (and others): Backbeats on 2 and 4
        backbeats = [steps_per_beat * 1, steps_per_beat * 3]
        
    for i in range(total_steps):
        human_shift = random.uniform(-jitter_amount, jitter_amount)
        start_t = max(0, (i * step_time) + human_shift)
        
        # Position within the current bar (e.g., 0 to 15 for 4/4)
        step_in_bar = i % steps_per_bar
        
        # 1. KICK (Pitch 36) - Downbeats
        is_beat = (step_in_bar % steps_per_beat == 0)
        kick_prob = 0.7 if is_beat else 0.1
        if random.random() < kick_prob:
            # Heaviest accent on the very first beat of the bar
            vel = random.randint(90, 127) if step_in_bar == 0 else random.randint(60, 90)
            drums.notes.append(pretty_midi.Note(vel, 36, start_t, start_t + 0.1))

        # 2. SNARE (Pitch 38) - Dynamic Backbeats
        is_backbeat = step_in_bar in backbeats
        snare_prob = 0.95 if is_backbeat else 0.05
        if random.random() < snare_prob:
            vel = random.randint(100, 127) if is_backbeat else random.randint(30, 60)
            drums.notes.append(pretty_midi.Note(vel, 38, start_t, start_t + 0.1))

        # 3. HI-HAT (Pitch 42) - Velocity accents on 8th notes
        if random.random() < 0.85:
            vel = random.randint(80, 100) if i % 2 == 0 else random.randint(40, 70)
            drums.notes.append(pretty_midi.Note(vel, 42, start_t, start_t + 0.05))

    pm.instruments.append(drums)
    
    # Ensure directory exists before saving
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    pm.write(filename)
    return filename

# --- GENERATION LOOP ---
# Let's generate 4 variations for 3 different time signatures
time_signatures = [(3, 4), (4, 4), (5, 4)]
kits_per_ts = 4

for ts_idx, (num, den) in enumerate(time_signatures):
    for kit_idx in range(kits_per_ts):
        # Naming convention: train_kit0_ts3-4.mid
        file_name = f'./midi/train_kit{kit_idx}_ts{num}-{den}.mid'
        generate_pro_midi(
            loop_length_bars=100, 
            tempo=120, 
            numerator=num, 
            denominator=den, 
            filename=file_name
        )
        print(f"Generated: {file_name}")

Generated: ./midi/train_kit0_ts3-4.mid
Generated: ./midi/train_kit1_ts3-4.mid
Generated: ./midi/train_kit2_ts3-4.mid
Generated: ./midi/train_kit3_ts3-4.mid
Generated: ./midi/train_kit0_ts4-4.mid
Generated: ./midi/train_kit1_ts4-4.mid
Generated: ./midi/train_kit2_ts4-4.mid
Generated: ./midi/train_kit3_ts4-4.mid
Generated: ./midi/train_kit0_ts5-4.mid
Generated: ./midi/train_kit1_ts5-4.mid
Generated: ./midi/train_kit2_ts5-4.mid
Generated: ./midi/train_kit3_ts5-4.mid


# Spectrogram generation

In [32]:
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Folders
AUDIO_FOLDER = "raw_audio/"
IMG_FOLDER = "dataset/images"
os.makedirs(IMG_FOLDER, exist_ok=True)

SR = 22050
SEGMENT_LENGTH = 2.0

def apply_spec_augment(spec):
    """
    Randomly masks horizontal (frequency) and vertical (time) strips.
    This is the ultimate 'anti-cheat' for ML models.
    """
    spec = spec.copy()
    num_mask = 2
    # Frequency masking
    for i in range(num_mask):
        f = np.random.randint(0, 15) # Width of the mask
        f0 = np.random.randint(0, 128 - f)
        spec[f0:f0 + f, :] = np.min(spec)
    return spec

labels = []

for wav_file in os.listdir(AUDIO_FOLDER):
    if not wav_file.endswith(".wav"): continue

    kit_name = wav_file.replace(".wav", "")
    y, sr = librosa.load(os.path.join(AUDIO_FOLDER, wav_file), sr=SR)
    y = y / (np.max(np.abs(y)) + 1e-6)

    num_segments = int(len(y) / (SEGMENT_LENGTH * sr))

    for i in range(num_segments):
        start = int(i * SEGMENT_LENGTH * sr)
        end = int((i + 1) * SEGMENT_LENGTH * sr)
        segment = y[start:end]
        
        # --- THE FIXES ---

        # 1. Subtle Pitch Shifting (+/- 0.5 semitone)
        # This shifts those 'hot rows' in your heatmap up and down.
        steps = np.random.uniform(-0.5, 0.5)
        segment = librosa.effects.pitch_shift(segment, sr=sr, n_steps=steps)

        # 2. Add White Noise Floor
        segment += 0.005 * np.random.randn(len(segment))

        # 3. Time Jitter
        segment = np.roll(segment, np.random.randint(-1000, 1000))

        # Generate Spectrogram
        S = librosa.feature.melspectrogram(y=segment, sr=sr, n_mels=128)
        S_db = librosa.power_to_db(S, ref=1.0)

        # 4. Apply SpecAugment (Frequency Masking)
        # This deletes random frequency bands so the model can't rely on them.
        S_db = apply_spec_augment(S_db)

        # Save
        img_file = f"img_{i:03d}.png"
        plt.figure(figsize=(6,4))
        librosa.display.specshow(S_db, sr=sr)
        plt.axis("off")
        plt.savefig(os.path.join(IMG_FOLDER+f'/{kit_name}/', img_file), bbox_inches="tight", pad_inches=0)
        plt.close()

        labels.append({"filename": img_file, "label": kit_name})

pd.DataFrame(labels).to_csv("dataset/labels.csv", index=False)
print("Dataset hardened.")

Dataset hardened.
